# 01 — Isolated eager CUDA efficiency benchmark

Runs **FP32 eager** and **TorchAO dynamic W8A8 INT8 eager** in **separate fresh Python processes**. This removes the shared-model memory confound. The same HANS inputs, batch sizes, and measurement rules are applied to both models.

## Locate tools and verify defaults

The code requires the entire `runtime_analysis/` folder, but no edits to old notebooks.

In [1]:
from pathlib import Path
import subprocess, sys, json

def find_analysis_folder():
    current = Path.cwd().resolve()
    for parent in [current, *current.parents]:
        for relative in [Path('notebooks/runtime_analysis'), Path('runtime_analysis'), Path('.')]:
            candidate = (parent / relative).resolve()
            if (candidate / 'config.json').is_file() and (candidate / 'worker.py').is_file():
                return candidate
    raise FileNotFoundError('Cannot locate notebooks/runtime_analysis/config.json. Open this notebook inside the copied repository.')

ANALYSIS_DIR = find_analysis_folder()
REPO_ROOT = ANALYSIS_DIR.parent.parent
CONFIG = json.loads((ANALYSIS_DIR/'config.json').read_text())
print('Folder:', ANALYSIS_DIR)
print('Repository:', REPO_ROOT)

Folder: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/notebooks/runtime_analysis
Repository: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers


## 🔴 Only change this if deliberately extending to other seeds

Start with `SEEDS = [42]`. Once seed 42 is validated, change it to `[123, 456]` or `[42, 123, 456]` here. The scripts use `config.json` for path and benchmark settings.

In [2]:
SEEDS = [42]  # 🔴 FIRST RUN = [42]; only afterwards run other seeds.
VARIANTS = ['fp32_eager', 'int8_eager']
cmd = [sys.executable, str(ANALYSIS_DIR / 'run_experiments.py'), '--seeds', *map(str, SEEDS), '--task', 'benchmark', '--variants', *VARIANTS]
print('Executing:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

Executing: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/.venv/bin/python /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/notebooks/runtime_analysis/run_experiments.py --seeds 42 --task benchmark --variants fp32_eager int8_eager
CUDA variants run SEQUENTIALLY in a FRESH PROCESS each.
Working directory: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers

Running seed=42, variant=fp32_eager, task=benchmark



W1025 22:45:56.968000 44475 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1025 22:45:56.985000 44475 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 23045.00it/s]


  fp32_eager repeat 1: 30000 examples, 9.879s, 3036.7/s
  fp32_eager repeat 2: 30000 examples, 10.031s, 2990.8/s
  fp32_eager repeat 3: 30000 examples, 10.198s, 2941.8/s
Saved: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/results/runtime_analysis/seed42/fp32_eager/benchmark_summary.json
Finished exit code 0 in 36.8s

Running seed=42, variant=int8_eager, task=benchmark



W1025 22:46:33.851000 44636 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1025 22:46:33.868000 44636 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 16826.10it/s]


  int8_eager repeat 1: 30000 examples, 54.189s, 553.6/s
  int8_eager repeat 2: 30000 examples, 53.368s, 562.1/s
  int8_eager repeat 3: 30000 examples, 53.118s, 564.8/s
Saved: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/results/runtime_analysis/seed42/int8_eager/benchmark_summary.json
Finished exit code 0 in 173.3s

All requested subprocess jobs completed successfully.


CompletedProcess(args=['/home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/.venv/bin/python', '/home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/notebooks/runtime_analysis/run_experiments.py', '--seeds', '42', '--task', 'benchmark', '--variants', 'fp32_eager', 'int8_eager'], returncode=0)

## Save combined CSV comparisons

In [3]:
subprocess.run([sys.executable, str(ANALYSIS_DIR/'summarize_results.py')], cwd=REPO_ROOT, check=True)
import pandas as pd
out_dir = REPO_ROOT / CONFIG['results_dir']
display(pd.read_csv(out_dir / 'paired_fp32_int8_comparison.csv'))

Matched comparisons:
  seed                mode fp32_variant int8_variant  int8_over_fp32_throughput_x
   42 eager_original_int8   fp32_eager   int8_eager                     0.187956
Saved: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/results/runtime_analysis/all_benchmark_variants.csv
Saved: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/results/runtime_analysis/benchmark_inclusion_status.csv
Saved: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/results/runtime_analysis/pair_eligibility_status.csv
NOTE: FP32-head INT8 is a DISTINCT quantization variant; check 04 prediction equivalence before relating to original HANS behavioral results.


,seed,mode,fp32_variant,int8_variant,comparison_note,throughput_batch_size,num_hans_examples,partial_batch_policy,compile_mode,device,...,int8_batch32_median_forward_latency_ms,int8_change_pct_batch32_median_forward_latency_ms,fp32_batch32_p95_forward_latency_ms,int8_batch32_p95_forward_latency_ms,int8_change_pct_batch32_p95_forward_latency_ms,fp32_runtime_quantization_setup_seconds,int8_runtime_quantization_setup_seconds,int8_change_pct_runtime_quantization_setup_seconds,int8_over_fp32_throughput_x,int8_inference_speedup_x
0,42,eager_original_int8,fp32_eager,int8_eager,Original W8A8 dynamic INT8 versus FP32; same e...,32,30000,repeat examples within incomplete final batch ...,NaN,cuda,...,58.711331,471.241007,12.827604,59.889633,366.88091,0.0,0.180217,NaN,0.187956,0.187956


## What to interpret

- **Primary:** serialized state size; model-resident CUDA allocation delta; peak allocated CUDA memory; batch-1 median/p95 *forward-only* latency; batch-32 *end-to-end loop* throughput.
- **Supporting:** full-HANS time and quantization setup time.
- **Not a valid model-memory comparison:** shared-process RSS or logical `numel*element_size()` for TorchAO tensor subclasses.

Next: **`02_cuda_profiler.ipynb`** to understand why eager INT8 was slow.